## Data Ingestion for OCI ADK AgenticRag

## Ingest PDF Document to Oracle 23ai vector store

### Import Libraries

In [ ]:
from pypdf import PdfReader
import oracledb
import oci
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.embeddings import OCIGenAIEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import BaseDocumentTransformer,Document
from LoadProperties import LoadProperties
print("Successfully imported libraries and modules")

#

we use LoadProperties class to read config.txt and
load model name, service endpoint, embedding model name and
compartment ocid. Next, we declare username , password and database
connection string. Finally, we create a database connection using username,
password and connection string.

In [ ]:
properties = LoadProperties()
signer = oci.auth.signers.InstancePrincipalsSecurityTokenSigner()

un = "vector"
pw = "vector"
cs = "localhost/FREEPDB1"

import oracledb
try:
    conn3c = oracledb.connect(user=un,password=pw,dsn=cs)
    print("Connection successful !!")
except Exception as e :
    print("Connection failed !!")
    

#

we extract text of all the pages of a PDF which is oci-aifoundations.
pdf in this case and split it into chunks of 2000 characters with the
overlap of 100 characters. This ensures that chunks do not loose continuity of the
context.

In [ ]:
# RAG Step1 Load the PDF document and create pdf reader object

pdf = PdfReader('./pdf-docs/oci-ai-foundations.pdf')

# RAG step2 Transform the document to text 
text = ""

for page in pdf.pages:
    text += page.extract_text()

print("You have transformed the PDF document to text format")

# RAG step3 Chunk the text into smaller chunks

text_splitter = CharacterTextSplitter(separator=".", chunk_size=2000, chunk_overlap=100)

chunks = text_splitter.split_text(text)


#

we declare chunks_to_docs_wrapper function to add
metadata to the chunks and convert the chunk text and meta data to Document object.
We iterate through all the chunks, extract page number and text from each chunk and
set these in the metadata.

In [ ]:
# Function to format and add metadata to Oracle 23ai Vector store

def chunks_to_docs_wrapper(row:dict) -> Document:
    metadata={'id':row['id'], 'link':row['link']}
    return Document(page_content=row['text'], metadata=metadata)

# RAG step4 create metadata wrapper to store additional information in vector store

docs = [chunks_to_docs_wrapper({'id':str(page_num),'link':
                               f'Page {page_num}', 'text':text}) for page_num, text in enumerate(chunks)]    

#

we use OCIGenAIEmbeddings to create an embed_model
object. We pass in model name, service endpoint and compartment ocid. We have set
policies for our VM to access OCI Generative AI service, hence we use auth_type as
INSTANCE_PRINCIPAL. Next, we embed and save all the docs in the DEMO_TABLE of
the Oracle 23 ai database using from_documents method of the OracleVS class. We will
use DOT_PRODUCT to compute similarity between embeddings.

In [ ]:
# RAG step5 using an embedding model embed the chunks as vetors into oracle database 23ai

embed_model = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',
)

#

we use OCIGenAIEmbeddings to create an embed_model
object. We pass in model name, service endpoint and compartment ocid. We have set
policies for our VM to access OCI Generative AI service, hence we use auth_type as
INSTANCE_PRINCIPAL. Next, we embed and save all the docs in the DEMO_TABLE of
the Oracle 23 ai database using from_documents method of the OracleVS class. We will
use DOT_PRODUCT to compute similarity between embeddings.

In [ ]:
# RAG step6 configure the vector store with the model , table name and using indicated distance 
# strategy for the similarity search and vectorize the chunks

knowledge_base = OracleVS.from_documents(docs, embed_model,client=conn3c,
                                        table_name="DEMO_TABLE",
                                        distance_strategy=DistanceStrategy.DOT_PRODUCT)

print("Chunks are stored in the DEMO_TABLE")